# Demo — 静かな変化 × 類似企業

LangGraph orchestrator を起動し、レポートと可視化を表示します。

In [ ]:
import sys, pathlib
ROOT = pathlib.Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

from app.graph import build_graph
graph = build_graph()
final = graph.invoke({})
print('companies analyzed:', len(final.get('fundamentals', {})))

In [ ]:
from IPython.display import Markdown, display
display(Markdown(final['report']))

## セグメント系列 × テンプレート 可視化

ワールド (3612) のプラットフォーム構成比の推移をテンプレートとし、候補企業の系列を重ね描きします。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from app.tools.dtw_match import zscore

fund = final['fundamentals']
tpl_series = fund['3612']['segment_history']['プラットフォーム']

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(zscore(tpl_series), marker='o', linewidth=2.5, label='ワールド 3612 PF (template)')
for code in ('3923', '3697', '4477'):
    hist = fund.get(code, {}).get('segment_history', {})
    if not hist:
        continue
    seg_name, series = max(hist.items(), key=lambda kv: len(kv[1]))
    ax.plot(zscore(series), marker='.', linestyle='--', label=f'{code} {seg_name}')
ax.set_title('セグメント構成比 — z-scored 推移')
ax.set_xlabel('quarter index')
ax.set_ylabel('z-score')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
pd.DataFrame(final['similar_company_results']).head(10)

In [ ]:
pd.DataFrame(final['quiet_change_results'])